# FinRAG India
### Financial Intelligence RAG for Indian Markets

**60 documents** from RBI, SEBI, NSE, BSE, Ministry of Finance, Economic Times, Business Standard, Mint, Moneycontrol, CRISIL, ICRA and other credible Indian sources.

---

**Run cells 1, 2, 3 in order.** A public URL appears at the end of Cell 3.

| Login | Password | Daily queries |
|-------|----------|---------------|
| `demo` | `demo123` | 25 |
| `guest` | `guest123` | 15 |
| `anish` | `anish2025` | 100 |
| `guide` | `sampada25` | 100 |
| `admin` | `admin123` | 200 |

In [ ]:
# ============================================================
# CELL 1  --  Install dependencies
# ============================================================
!pip install -q streamlit pandas
print('Dependencies installed')

In [ ]:
# ============================================================
# CELL 2  --  Write app.py  (60 Indian market documents)
# ============================================================
# The original Colab notebook embedded app.py as a base64 string.
# This repo copy fetches the identical app.py (63,398 bytes) from GitHub instead.
import os, urllib.request
if not os.path.exists('app.py'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/anishkaranth/bits-finrag-final-project/main/app.py', 'app.py')
print('app.py written:', os.path.getsize('app.py'), 'bytes')


In [ ]:
# ============================================================
# CELL 3  --  Launch the app
# ============================================================
import subprocess, threading, time, re, os

subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

def run_app():
    subprocess.run([
        'streamlit', 'run', 'app.py',
        '--server.port', '8501',
        '--server.headless', 'true',
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false',
        '--server.address', '0.0.0.0',
        '--theme.base', 'light',
        '--theme.primaryColor', '#0f766e',
    ])

threading.Thread(target=run_app, daemon=True).start()
print('Starting Streamlit...')
time.sleep(6)

if not os.path.exists('./cloudflared'):
    print('Downloading tunnel...')
    subprocess.run(['wget', '-q', '-O', 'cloudflared',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'],
        capture_output=True)
    subprocess.run(['chmod', '+x', 'cloudflared'])

proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

print('Waiting for public URL...')
for line in proc.stdout:
    m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if m:
        print()
        print('=' * 62)
        print('  FinRAG India is LIVE')
        print()
        print('  URL:', m.group())
        print()
        print('  Login with:   demo / demo123')
        print('  or:           anish / anish2025')
        print('=' * 62)
        break

---
### Alternative: open inside the notebook

In [ ]:
# ============================================================
# CELL 4  --  Open inside Colab (alternative to the URL above)
# ============================================================
from IPython.display import IFrame, display
display(IFrame(src='http://localhost:8501', width='100%', height=820))

---
### Admin utility

In [ ]:
# ============================================================
# CELL 5  --  (Optional) Reset all user token quotas
# ============================================================
import json, os
from datetime import date

if os.path.exists('users.json'):
    with open('users.json') as f:
        users = json.load(f)
    for u in users:
        users[u]['used_today'] = 0
        users[u]['last_date'] = str(date.today())
    with open('users.json', 'w') as f:
        json.dump(users, f, indent=2)
    print('All quotas reset for', len(users), 'users')
else:
    print('No users.json yet - run the app first')